# Dashboard: will a driver score points?

This app **loads** what the training stage produced (`artifacts/`) and lets you explore it. It **never retrains anything**.

Three views, matching the business question (a fantasy-F1 platform recommending drivers likely to finish in the top 10):

1. **Model comparison**: the three models side by side, with a predicted-vs-actual plot.
2. **Distributions**: how each feature and the target look, split by actual outcome.
3. **Threshold & cost**: move a decision threshold and watch the confusion matrix and the business cost change.

In [ ]:
import json
from pathlib import Path

import gradio as gr
import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
import torch.nn as nn

## 1. Load the artifacts

Everything comes from files saved by `train.ipynb`: the three models, the preprocessing statistics (median, mean, std), the configuration (including the costs from the business framing), the results table, and a table of every driver-race with its split and true label.

If a file is missing the app stops with a clear message instead of silently retraining.

In [ ]:
ARTIFACTS = Path("artifacts")
REQUIRED = ["sklearn_model.joblib", "manual_torch.pt", "nn_module.pt",
            "preprocess.json", "config.json", "metrics.json", "predictions.csv"]
missing = [f for f in REQUIRED if not (ARTIFACTS / f).exists()]
if missing:
    raise FileNotFoundError(f"Missing artifacts {missing}. Run `uv run python main.py train` first.")

config = json.loads((ARTIFACTS / "config.json").read_text())
prep = json.loads((ARTIFACTS / "preprocess.json").read_text())
metrics = pd.read_json(ARTIFACTS / "metrics.json")
saved = pd.read_csv(ARTIFACTS / "predictions.csv")
FEATURES = prep["features"]

# the nn.Module class must match the one used in training so the saved weights fit
class LogisticModel(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x).squeeze(1)

sk_model = joblib.load(ARTIFACTS / "sklearn_model.joblib")
manual = torch.load(ARTIFACTS / "manual_torch.pt", weights_only=True)
net = LogisticModel(len(FEATURES))
net.load_state_dict(torch.load(ARTIFACTS / "nn_module.pt", weights_only=True))
net.eval()

print("loaded 3 models,", len(saved), "driver-races,", len(FEATURES), "features")

## 2. Predict with the loaded models

We apply the **same preprocessing as in training** (fill with the training median, standardise with the training mean and std) and then score every row with each loaded model. This is inference only: no fitting happens here.

As a check, the probabilities we recompute are compared with the ones saved during training. They should match almost exactly.

In [ ]:
def transform(frame):
    median = pd.Series(prep["median"])
    mean = pd.Series(prep["mean"])
    std = pd.Series(prep["std"])
    return ((frame[FEATURES].fillna(median) - mean) / std).values.astype(np.float32)

X = transform(saved)
with torch.no_grad():
    p_sklearn = sk_model.predict_proba(X)[:, 1]
    p_manual = torch.sigmoid(torch.tensor(X) @ manual["w"] + manual["b"]).numpy()
    p_nn = torch.sigmoid(net(torch.tensor(X))).numpy()

data = saved.copy()
data["p_sklearn"], data["p_manual"], data["p_nn"] = p_sklearn, p_manual, p_nn

# sanity check against what training saved
for col in ["p_sklearn", "p_manual", "p_nn"]:
    diff = float(np.abs(saved[col].values - data[col].values).max())
    print(f"{col}: max difference vs saved predictions = {diff:.2e}")
    if diff > 1e-4:
        print("  WARNING: loaded model disagrees with saved predictions")

## 3. Shared helpers

- `MODELS` maps the names shown in the dropdowns to the probability columns.
- `confusion_counts` turns probabilities and a threshold into TP / FP / FN / TN.
- `expected_cost` is the same formula as in training: `(FP * cost_FP + FN * cost_FN) / n`. The default costs come from `config.json`, so they match the business framing.

In [ ]:
SPLITS = ["train", "val", "test"]
MODELS = {
    "scikit-learn": ("p_sklearn", "sklearn LogisticRegression"),
    "PyTorch manual loop": ("p_manual", "PyTorch manual loop"),
    "PyTorch nn.Module + optim": ("p_nn", "PyTorch nn.Module + optim"),
}
COLORS = {"scikit-learn": "#1f77b4", "PyTorch manual loop": "#ff7f0e", "PyTorch nn.Module + optim": "#2ca02c"}
COST_FP, COST_FN = config["cost_fp"], config["cost_fn"]
TEMPLATE = "plotly_white"

def subset(split):
    return data if split == "all" else data[data["split"] == split]

def confusion_counts(y, p, thr):
    pred = p >= thr
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    return tp, fp, fn, tn

def expected_cost(y, p, thr, c_fp, c_fn):
    _, fp, fn, _ = confusion_counts(y, p, thr)
    return (fp * c_fp + fn * c_fn) / len(y)

def val_threshold(model_name):
    # the threshold chosen on the validation set during training
    row = metrics[metrics["model"] == MODELS[model_name][1]]
    return float(row["threshold"].iloc[0])

## 4. View 1: model comparison

- **Predicted vs actual (calibration):** predictions are grouped into 10 probability bins. For each bin we plot the average predicted probability against the share of drivers who really scored. A model that is well calibrated lies on the diagonal.
- **Agreement:** each point is one driver-race; if two implementations are equivalent they fall on the diagonal.
- The results table is the one computed on the **test set** during training.

In [ ]:
def calibration_plot(split):
    d = subset(split)
    y = d["scores_points"].values
    bins = np.linspace(0, 1, 11)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="perfect",
                             line=dict(color="gray", dash="dash")))
    for name, (col, _) in MODELS.items():
        p = d[col].values
        idx = np.digitize(p, bins[1:-1])
        xs, ys = [], []
        for b in range(10):
            m = idx == b
            if m.sum() >= 5:
                xs.append(p[m].mean())
                ys.append(y[m].mean())
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines+markers", name=name,
                                 line=dict(color=COLORS[name])))
    fig.update_layout(template=TEMPLATE, title=f"Predicted vs actual ({split} set)",
                      xaxis_title="Average predicted probability of scoring",
                      yaxis_title="Share that actually scored", height=420)
    return fig

def agreement_plot(split):
    d = subset(split)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="identical",
                             line=dict(color="gray", dash="dash")))
    for name in ["PyTorch manual loop", "PyTorch nn.Module + optim"]:
        fig.add_trace(go.Scatter(x=d["p_sklearn"], y=d[MODELS[name][0]], mode="markers", name=name,
                                 marker=dict(size=4, opacity=0.5, color=COLORS[name])))
    fig.update_layout(template=TEMPLATE, title=f"Do the implementations agree? ({split} set)",
                      xaxis_title="scikit-learn probability", yaxis_title="PyTorch probability", height=420)
    return fig

def comparison_views(split):
    return calibration_plot(split), agreement_plot(split)

metrics_view = metrics.copy()
metrics_view.columns = [c.replace("@", " @ ") for c in metrics_view.columns]

## 5. View 2: distributions

- **Feature distribution:** pick a feature and a split; the histogram shows drivers who scored and drivers who did not. If the two overlap a lot, the feature carries little signal.
- **Target by season:** the share of drivers who scored in each season, coloured by the split they belong to. It is near 50% because points go to 10 of about 20 starters.

In [ ]:
def feature_hist(feature, split):
    d = subset(split)
    fig = go.Figure()
    for label, name, color in [(1, "scored points", "#2ca02c"), (0, "no points", "#d62728")]:
        vals = d.loc[d["scores_points"] == label, feature].dropna()
        fig.add_trace(go.Histogram(x=vals, name=name, opacity=0.6, marker_color=color))
    fig.update_layout(template=TEMPLATE, barmode="overlay", height=420,
                      title=f"{feature} by outcome ({split} set)",
                      xaxis_title=feature, yaxis_title="Driver-races")
    return fig

def target_by_season():
    g = data.groupby(["year", "split"], as_index=False)["scores_points"].mean()
    colors = {"train": "#1f77b4", "val": "#ff7f0e", "test": "#2ca02c"}
    fig = go.Figure()
    for s in SPLITS:
        part = g[g["split"] == s]
        fig.add_trace(go.Bar(x=part["year"], y=part["scores_points"], name=s, marker_color=colors[s]))
    fig.update_layout(template=TEMPLATE, title="Share of drivers who scored, by season", height=420,
                      xaxis_title="Season", yaxis_title="Share scoring", yaxis_range=[0, 1])
    return fig

## 6. View 3: threshold and business cost

The model gives a probability; the platform recommends a driver only if it is at least the **threshold**. Lowering it recommends more drivers (fewer missed scorers, more wrong recommendations); raising it does the opposite.

- The **confusion matrix** shows the four outcomes at the chosen threshold.
- The **cost** is `(FP x cost_FP + FN x cost_FN) / n`, with the costs from the business framing (the sliders start at those values). It is compared with the naive baseline "scores if starting in the top 10".
- The **cost curve** shows the cost at every threshold. The threshold used in the report was chosen on the **validation** set; looking at the test set here is for evaluation only.

In [ ]:
def threshold_view(model_name, split, thr, c_fp, c_fn):
    d = subset(split)
    y = d["scores_points"].values
    p = d[MODELS[model_name][0]].values
    tp, fp, fn, tn = confusion_counts(y, p, thr)
    n = len(y)
    cost = (fp * c_fp + fn * c_fn) / n
    base_pred = d["baseline_pred"].values
    base_cost = ((((base_pred == 1) & (y == 0)).sum() * c_fp) + (((base_pred == 0) & (y == 1)).sum() * c_fn)) / n

    # confusion matrix
    cm = go.Figure(go.Heatmap(
        z=[[tp, fn], [fp, tn]],
        x=["Predicted: scores", "Predicted: no points"],
        y=["Actual: scores", "Actual: no points"],
        colorscale="Blues", showscale=False, texttemplate="%{z}", textfont=dict(size=22)))
    cm.update_yaxes(autorange="reversed")
    cm.update_layout(template=TEMPLATE, title=f"Confusion matrix at threshold {thr:.2f}", height=380)

    # cost curve over all thresholds
    grid = np.linspace(0.01, 0.99, 99)
    costs = [expected_cost(y, p, t, c_fp, c_fn) for t in grid]
    best = grid[int(np.argmin(costs))]
    curve = go.Figure()
    curve.add_trace(go.Scatter(x=grid, y=costs, mode="lines", name="model cost"))
    curve.add_hline(y=base_cost, line_dash="dash", line_color="gray", annotation_text="baseline (grid <= 10)")
    curve.add_vline(x=thr, line_color="crimson", annotation_text="chosen")
    curve.update_layout(template=TEMPLATE, title=f"Cost per driver-race vs threshold ({split} set)",
                        xaxis_title="Threshold", yaxis_title="Expected cost", height=380)

    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    saving = (base_cost - cost) / base_cost * 100 if base_cost else float("nan")
    summary = (
        f"### Cost per driver-race: **{cost:.3f}**  (baseline: {base_cost:.3f}; {abs(saving):.1f}% {'cheaper' if saving >= 0 else 'more expensive'} than the baseline)\n\n"
        f"- Total cost: **{fp * c_fp + fn * c_fn:.0f}**  = {fp} false positives x {c_fp} + {fn} false negatives x {c_fn}\n"
        f"- Precision: {precision:.3f} | Recall: {recall:.3f} | Accuracy: {(tp + tn) / n:.3f}\n"
        f"- Threshold chosen on validation during training: **{val_threshold(model_name):.2f}** "
        f"| cheapest threshold on this split: {best:.2f}"
    )
    return cm, curve, summary

## 7. Build the interface

A `gr.Blocks` layout with one tab per view. Each control is connected to the function that redraws the plots; `demo.load` draws everything once when the page opens. The threshold slider starts at the validation-optimal threshold saved by training.

In [ ]:
DEFAULT_THR = val_threshold("scikit-learn")
FEATURE_CHOICES = FEATURES

with gr.Blocks(title="F1: will a driver score points?") as demo:
    gr.Markdown(
        "# Will a driver score points?\n"
        "A fantasy-F1 platform recommends drivers likely to finish in the top 10. "
        "Predictions are made **after qualifying and before the race**. "
        "All models are loaded from `artifacts/`; nothing is retrained here."
    )

    with gr.Tab("Model comparison"):
        gr.Markdown("Results on the **test set** (2022-2024), computed during training. Lower cost is better.")
        gr.Dataframe(value=metrics_view, interactive=False)
        split_cmp = gr.Dropdown(SPLITS, value="test", label="Data split for the plots")
        with gr.Row():
            cal_plot = gr.Plot()
            agr_plot = gr.Plot()
        split_cmp.change(comparison_views, split_cmp, [cal_plot, agr_plot])

    with gr.Tab("Distributions"):
        with gr.Row():
            feat = gr.Dropdown(FEATURE_CHOICES, value="grid", label="Feature")
            split_dist = gr.Dropdown(["all"] + SPLITS, value="all", label="Data split")
        hist_plot = gr.Plot()
        season_plot = gr.Plot(value=target_by_season())
        feat.change(feature_hist, [feat, split_dist], hist_plot)
        split_dist.change(feature_hist, [feat, split_dist], hist_plot)

    with gr.Tab("Threshold & cost"):
        with gr.Row():
            model_dd = gr.Dropdown(list(MODELS), value="scikit-learn", label="Model")
            split_thr = gr.Dropdown(SPLITS, value="test", label="Data split")
        thr = gr.Slider(0.05, 0.95, value=DEFAULT_THR, step=0.01, label="Decision threshold")
        with gr.Row():
            c_fp = gr.Slider(0, 5, value=COST_FP, step=0.5, label="Cost of a false positive")
            c_fn = gr.Slider(0, 5, value=COST_FN, step=0.5, label="Cost of a false negative")
        summary_md = gr.Markdown()
        with gr.Row():
            cm_plot = gr.Plot()
            curve_plot = gr.Plot()
        thr_inputs = [model_dd, split_thr, thr, c_fp, c_fn]
        for control in thr_inputs:
            control.change(threshold_view, thr_inputs, [cm_plot, curve_plot, summary_md])

    # draw everything once when the page opens
    demo.load(comparison_views, split_cmp, [cal_plot, agr_plot])
    demo.load(feature_hist, [feat, split_dist], hist_plot)
    demo.load(threshold_view, thr_inputs, [cm_plot, curve_plot, summary_md])

## 8. Launch

`main.py` runs this notebook as a script, so `demo.launch()` starts the server and opens the app in the browser.

In [ ]:
demo.launch()